# Phase 06B.02 — Knowledge-Augmented VQA Preflight

Freeze authoritative corpus provenance and the controlled Static-K/RAG-k experiment matrix; this notebook does not fetch sources or evaluate validation.

**Immutable gates:** `L32-F1`; 298 frozen validation samples; train-side checkpoint selection only; no public-test access. Missing human/input artifacts produce an explicit status and stop—no synthetic labels or provenance.

## 1. Selected-track gate

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None
def write_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path
from phase06a_common import sha256_file,sha256_json
from phase06b_common import load_json,validate_knowledge_corpus_manifest
P=ROOT/"outputs/phase06b/novelty_protocol/novelty_protocol.json"; OUT=ROOT/"outputs/phase06b/knowledge_augmented/preflight"
CM=ROOT/"data/knowledge/traffic_law_corpus_manifest.json"; OUT.mkdir(parents=True,exist_ok=True)
if not P.is_file(): raise RuntimeError("Lock Phase06B_01 first")
protocol=load_json(P); selected=protocol.get("status")=="locked" and protocol.get("selected_track")=="knowledge_augmented"
if not selected: write_json(OUT/"PHASE06B_02_STATUS.json",{"status":"not_selected","selected_track":protocol.get("selected_track")})
print("selected:",selected)

## 2. Local source and hash gate

In [ ]:
if selected and not CM.is_file():
 write_json(OUT/"traffic_law_corpus_manifest.template.json",{"corpus_name":"Vietnam traffic-law corpus","version":"",
  "effective_date_cutoff":"2026-08-25","documents":[{"document_id":"","title":"","local_path":"data/knowledge/...",
  "source_url":"","issuing_authority":"","effective_date":"","sha256":"","license_or_access_note":""}]})
 write_json(OUT/"PHASE06B_02_STATUS.json",{"status":"awaiting_corpus_manifest"})
 raise RuntimeError("Selected RAG track requires a frozen authoritative corpus")
if selected:
 corpus=validate_knowledge_corpus_manifest(load_json(CM))
 for doc in corpus["documents"]:
  path=ROOT/doc["local_path"]
  if not path.is_file() or sha256_file(path)!=doc["sha256"]: raise ValueError(f"Corpus file/hash mismatch: {path}")

## 3. Freeze minimum matrix
Oracle-K is subset-only diagnostic and cannot be selected as winner.

In [ ]:
if selected:
 rag={"status":"locked","selected_track":"knowledge_augmented","parent_protocol_sha256":sha256_json(protocol),
 "corpus_manifest_sha256":sha256_file(CM),"control":"L32-F1","variants":{
 "L32-F1":{"knowledge":"none","winner_eligible":True},"Static-K":{"knowledge":"fixed","winner_eligible":True},
 "RAG-k":{"knowledge":"retrieved_top_k","winner_eligible":True},"Oracle-K":{"subset_only":True,"winner_eligible":False}},
 "retrieval_metrics":["Recall@k","MRR","failure_taxonomy"],"one_factor_at_a_time":["k","chunk_size","reranker","knowledge_on_off"],
 "forbidden":["validation correctness for passage selection","public-test access"]}
 write_json(OUT/"knowledge_augmented_protocol.json",rag)
 write_json(OUT/"PHASE06B_02_STATUS.json",{"status":"complete","protocol_sha256":sha256_json(rag)})
 rag